In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
report_path = "results/eval_report_20260401_105452.json"

eval_report = pd.read_json(f'../{report_path}')

In [ ]:
total_cases = len(eval_report)

safety_rate = (
  (eval_report['fch'] == False) & (eval_report['ich'] == False)
).sum() / total_cases * 100

fnr = (
  eval_report['fn'] == True
).sum() / total_cases * 100

avg_score = eval_report['grade'].mean()

print(f"--- Global Metrics ---")
print(f"Number of cases evaluated: {total_cases}")
print(f"Safety Rate (No FCH or ICH): {safety_rate:.1f}%")
print(f"False Negative Rate: {fnr:.1f}%")
print(f"Average Quality Score: {avg_score:.2f}/5.0")


In [ ]:
# split the category into complexity and language
eval_report[['complexity', 'formality']] = eval_report['category'].apply(lambda x: pd.Series(['', ''] if x=='out_of_domain' else x.split('_')))

ich_only = eval_report['ich'] & ~eval_report['fch']
eval_report.loc[ich_only, 'Hallucination'] = "ICH only"
fch_only = ~eval_report['ich'] & eval_report['fch']
eval_report.loc[fch_only, 'Hallucination'] = "FCH only"
both_h = eval_report['ich'] & eval_report['fch']
eval_report.loc[both_h, 'Hallucination'] = "Both FCH and ICH"
no_h = ~eval_report['ich'] & ~eval_report['fch']
eval_report.loc[no_h, 'Hallucination'] = "No hallucination"

in_domain = eval_report[eval_report['category'] != 'out_of_domain']

sns.catplot(in_domain, kind="bar", x="Hallucination", y="grade", col='complexity', hue='formality')
plt.show()